# 聚宽逐股行业成员导出

按历史日期获取股票池和行业成员，输出可由 `import-jq-industry --members` 导入。
API 可用性取决于账号和研究环境权限。没有成员 API 时停止导出，请改用本地 `export_joinquant_members_local.py`。
优先使用 `get_industry` 获取 L1/L2/L3，其次使用 `get_industry_stocks` 获取 L1。
默认每日快照；大于 1 天的采样无法精确定位快照之间的成员变更。
上传 notebook，修改日期，逐单元运行后在文件面板下载 `jq_members.json`。


In [ ]:
# ===== 参数配置 =====
# 单日快照模式：只导出一个日期的成员（与 get_industry 返回的"股票→行业信息"字典一致）
AS_OF = '2026-06-01'        # 例如回测起点；设为 None 则改用下面的区间模式

# 区间快照模式：覆盖整个回测区间，每行带 in_date/out_date，保留历史成员变更
START = '2026-04-01'        # 回测开始前更早的成员基准日
END   = '2026-07-31'        # 回测结束（含）
STEP_DAYS = 1               # 每日快照；较粗采样只适合近似研究

OUTPUT = 'jq_members.json'  # 输出文件名（在聚宽「文件」面板下载）
METHOD = 'auto'             # auto=自动选择；也可强制 jqdata / industry_stocks

In [ ]:
import datetime
import json

def _resolve_api(name: str):
    """按 全局函数 → 所属模块 的顺序解析 API，找不到返回 None。

    支持 ``jqdata.get_industries``、``jqfactor.get_factors`` 这类带模块名的
    写法；不带模块名时先查全局函数（研究环境注入到全局命名空间），再查
    ``jqdata`` 模块属性。
    """
    if "." in name:
        module, attr = name.split(".", 1)
        try:
            mod = __import__(module, fromlist=[attr])
        except ImportError:
            return None
        return getattr(mod, attr, None)
    if name in globals():
        return globals()[name]
    try:
        import jqdata
    except ImportError:
        return None
    return getattr(jqdata, name, None)


def probe_industry_apis() -> dict[str, bool]:
    return {
        "get_industry": _resolve_api("get_industry") is not None,
        "get_industry_stocks": _resolve_api("get_industry_stocks") is not None,
        "jqdata.get_industries": _resolve_api("jqdata.get_industries") is not None,
        "get_all_securities": _resolve_api("get_all_securities") is not None,
    }




In [ ]:
def all_stock_securities(as_of):
    get_all_securities = _resolve_api("get_all_securities")
    if get_all_securities is None:
        raise RuntimeError("研究环境没有 get_all_securities")
    frame = get_all_securities(types=["stock"], date=as_of)
    return sorted(str(code) for code in frame.index.tolist())


def sw_l1_codes(as_of) -> dict[str, str]:
    """返回 {行业代码: 行业名称}。优先 get_industries('sw_l1') 动态获取。"""
    get_industries = _resolve_api("jqdata.get_industries")
    if get_industries is not None:
        try:
            frame = get_industries("sw_l1", date=as_of)
            if frame is not None and not frame.empty and "name" in frame.columns:
                result = {}
                for code, row in frame.iterrows():
                    name = str(row.get("name") or "")
                    code = str(code).strip()
                    if code and name:
                        result[code] = name
                if result:
                    return result
        except Exception:
            pass
    return {}


def industry_snapshot(as_of: str, method: str = "auto", chunk: int = 800):
    """返回 ({security: {sw_l1: {...}, ...}}, 使用的API方法)。"""
    get_industry = _resolve_api("get_industry")
    if method in ("auto", "jqdata") and get_industry is not None:
        codes = all_stock_securities(as_of)
        mapping = {}
        for start in range(0, len(codes), chunk):
            info = get_industry(codes[start : start + chunk], date=as_of) or {}
            for security, levels in info.items():
                payload = {}
                for level in ("sw_l1", "sw_l2", "sw_l3"):
                    value = levels.get(level) if isinstance(levels, dict) else None
                    if isinstance(value, dict):
                        payload[level] = {
                            "industry_code": str(value.get("industry_code") or ""),
                            "industry_name": str(value.get("industry_name") or ""),
                        }
                if payload.get("sw_l1", {}).get("industry_code"):
                    mapping[str(security)] = payload
        if mapping:
            return mapping, "get_industry"

    get_industry_stocks = _resolve_api("get_industry_stocks")
    if method in ("auto", "industry_stocks") and get_industry_stocks is not None:
        mapping = {}
        for code, name in sw_l1_codes(as_of).items():
            stocks = get_industry_stocks(code, date=as_of) or []
            for security in stocks:
                mapping.setdefault(str(security), {})["sw_l1"] = {
                    "industry_code": code,
                    "industry_name": name,
                }
        if mapping:
            return mapping, "get_industries + get_industry_stocks"


    raise RuntimeError("研究环境没有可用的行业 API: " + str(probe_industry_apis()))




In [ ]:
if STEP_DAYS < 1:
    raise ValueError('STEP_DAYS must be positive')
if not AS_OF and datetime.date.fromisoformat(START) > datetime.date.fromisoformat(END):
    raise ValueError('START must not be after END')
if AS_OF:
    # 单日快照：与 parse_members 的 JSON-dict 格式一致
    payload, method = industry_snapshot(AS_OF, METHOD)
    print('单日快照 %s：%d 只股票（API：%s）' % (AS_OF, len(payload), method), flush=True)
else:
    # 区间快照：与 parse_members 的 JSON-row 格式一致，每行带 in_date/out_date
    dates = []
    cursor = datetime.date.fromisoformat(START)
    end_date = datetime.date.fromisoformat(END)
    while cursor <= end_date:
        dates.append(cursor.isoformat())
        cursor += datetime.timedelta(days=STEP_DAYS)
    if dates[-1] != END:
        dates.append(END)
    rows = []
    snapshots = []
    for as_of in dates:
        print('导出 %s ...' % as_of, flush=True)
        snapshot, method = industry_snapshot(as_of, METHOD)
        snapshots.append((as_of, snapshot))
    for index, (as_of, snapshot) in enumerate(snapshots):
        out_date = snapshots[index + 1][0] if index + 1 < len(snapshots) else (end_date + datetime.timedelta(days=1)).isoformat()
        for security, levels in snapshot.items():
            rows.append({
                'security': security,
                'sw_l1_code': levels.get('sw_l1', {}).get('industry_code', ''),
                'sw_l2_code': levels.get('sw_l2', {}).get('industry_code', ''),
                'sw_l3_code': levels.get('sw_l3', {}).get('industry_code', ''),
                'in_date': as_of.replace('-', ''),
                'out_date': out_date.replace('-', '') if out_date else '',
            })
    payload = rows
    print('区间导出：%d 条成员记录' % len(payload), flush=True)

with open(OUTPUT, 'w', encoding='utf-8') as handle:
    json.dump(payload, handle, ensure_ascii=False, indent=1)
print('已写入：', OUTPUT)

In [ ]:
import os

size = os.path.getsize(OUTPUT) if os.path.exists(OUTPUT) else 0
print('文件大小：%.2f MB' % (size / 1024 / 1024))
if AS_OF:
    for security, levels in list(payload.items())[:3]:
        l1 = levels.get('sw_l1', {})
        print(security, l1.get('industry_code'), l1.get('industry_name'))

print('\n下一步：在聚宽左侧「文件」面板下载 %s，然后本地执行：' % OUTPUT)
if AS_OF:
    print('  python -m jq_tushare_sdk.cli import-jq-industry \\')
    print('    --classify joinquant_industry.txt --members %s \\' % OUTPUT)
    print('    --as-of %s --cache-db data/jq_tushare_cache.db' % AS_OF)
else:
    print('  python -m jq_tushare_sdk.cli import-jq-industry \\')
    print('    --classify joinquant_industry.txt --members %s \\' % OUTPUT)
    print('    --cache-db data/jq_tushare_cache.db')